# Module 6: Cleaning and Transforming Data

**ALY 6420 | Conditional Logic, NULL Handling, Text, Types, and Dates**

*Course Lecture Notes*


## Module 6

### Turning stored values into analysis-ready data

Through Module 5, you learned how to retrieve and assemble data. The next analytical problem is different: the correct rows may already be present, but the values are not yet in the form needed for analysis.

Typical preparation tasks include:

- translating codes into meaningful categories,
- handling missing or placeholder values,
- standardizing inconsistent text,
- converting values to the correct data type,
- deriving useful information from dates and timestamps, and
- combining several transformations into a clear preparation pipeline.

The emphasis in this module is **transformation without losing sight of meaning**. A technically valid expression can still produce a misleading dataset if the business rule behind it is wrong.


## Learning objectives

By the end of this lecture, you should be able to:

- use searched and simple `CASE` expressions to create conditional output
- explain why `CASE` condition order and `ELSE` behavior matter
- use `COALESCE` and `NULLIF` to control missing and exceptional values
- use `GREATEST` and `LEAST` to compare or bound values across a row
- standardize text with common PostgreSQL string functions
- convert compatible data types with `CAST()` and PostgreSQL's `::` shorthand
- explain why invalid or implicit conversions can fail
- derive useful fields from dates and timestamps
- use `DISTINCT` deliberately rather than as a substitute for diagnosing duplicate rows
- chain transformations into readable, testable steps before downstream aggregation or reporting


## Required preparation

### Principal resource

- Shan, J., Li, H., Goldwasser, M., Malik, U., & Johnston, B. (2025). *SQL for Data Analytics* (4th ed.), **Chapter 6: Transforming and Updating Data**.

The chapter motivates data transformation as the process of converting raw query output into more useful analytical values. It introduces `CASE WHEN`, date/time functions, string concatenation, casting, `COALESCE`, and `NULLIF` as core transformation tools.

### Practice environment

Run the examples against the **Pagila** PostgreSQL database in DBeaver. The examples primarily use `film`, `customer`, `payment`, and `rental`.

For every example, ask two questions before execution:

1. What will the expression return for a typical row?
2. What will it return for a missing, malformed, or boundary-case row?


# Part 1: Transformation as an Analytical Step


## Raw data can be correct and still be unusable

A stored value may be perfectly valid for the source system but inconvenient for analysis.

Examples:

| Stored value | Analytical need | Transformation |
|---|---|---|
| `4.99` | label rental price tier | `CASE` |
| `NULL` email | readable report value | `COALESCE` |
| empty string | treat as missing | `NULLIF` |
| `'  mary  '` | standardized name | `TRIM` + `INITCAP` |
| `'2026-09-05'` text | actual date | `CAST(... AS date)` |
| timestamp | reporting month | `DATE_TRUNC('month', ...)` |

The goal is not simply to change values. The goal is to make the dataset express the business concept you intend to analyze.


## Transformation should preserve the grain

Most functions in this module operate **row by row**.

If the input has one row per film, then this query still has one row per film:

```sql
SELECT film_id,
       title,
       UPPER(title) AS title_upper
FROM film;
```

No rows were grouped or duplicated. Only the representation of a value changed.

> **Grain check:** Before and after a cleaning step, be able to say what one row represents.


## Three questions before you transform a column

Before writing a function, identify:

1. **Current meaning** — what does the stored value mean?
2. **Desired meaning** — what should the new value represent?
3. **Exception behavior** — what should happen to `NULL`, empty strings, zeros, unexpected categories, or invalid text?

Most cleaning errors come from skipping the third question.


## Quick check 1

**Think before opening the answer.**

A report uses `COALESCE(email, 'no email')`. Does this change the stored value in the `customer` table?

<details>
<summary>Answer</summary>

No. In a `SELECT`, the expression changes only the query result. The underlying table remains unchanged unless you explicitly run a data-changing statement such as `UPDATE`.

</details>


# Part 2: Conditional Logic with CASE


## `CASE` turns business rules into values

Shan et al. introduce `CASE WHEN` as a conditional transformation: the database evaluates Boolean conditions and returns the value associated with the first condition that is true.

General searched form:

```sql
CASE
    WHEN condition_1 THEN result_1
    WHEN condition_2 THEN result_2
    ELSE result_other
END
```

`CASE` is an **expression**, so it can appear anywhere SQL expects a value: `SELECT`, `ORDER BY`, aggregates, and many other contexts.


## Example: turn rental rates into price tiers

```sql
SELECT film_id,
       title,
       rental_rate,
       CASE
           WHEN rental_rate < 1 THEN 'Budget'
           WHEN rental_rate < 4 THEN 'Standard'
           ELSE 'Premium'
       END AS price_tier
FROM film
ORDER BY rental_rate, title;
```

Read the logic from top to bottom.

```text
Is rate < 1?  ---- yes ---> Budget
     |
     no
     v
Is rate < 4?  ---- yes ---> Standard
     |
     no
     v
Premium
```


## First true condition wins

Condition order is part of the business rule.

This is correct:

```sql
CASE
    WHEN rental_rate < 1 THEN 'Budget'
    WHEN rental_rate < 4 THEN 'Standard'
    ELSE 'Premium'
END
```

This is wrong for the same intent:

```sql
CASE
    WHEN rental_rate < 4 THEN 'Standard'
    WHEN rental_rate < 1 THEN 'Budget'
    ELSE 'Premium'
END
```

A rate of `0.99` satisfies both conditions. In the second version, the broader `< 4` test appears first, so the row never reaches `< 1`.


## Boundary conditions need deliberate choices

Suppose a manager says:

> Films under 90 minutes are Short, films under 120 are Medium, and the rest are Long.

```sql
SELECT title,
       length,
       CASE
           WHEN length < 90 THEN 'Short'
           WHEN length < 120 THEN 'Medium'
           ELSE 'Long'
       END AS length_band
FROM film;
```

Ask what happens at exactly `90` and exactly `120`.

- `90` is **Medium**.
- `120` is **Long**.

That is not a SQL detail; it is the consequence of the business rule encoded with `<`.


## The `ELSE` clause controls unmatched rows

If no `WHEN` condition is true and no `ELSE` is provided, the result is `NULL`.

```sql
CASE
    WHEN rating = 'G' THEN 'General'
    WHEN rating = 'PG' THEN 'Guidance'
END
```

Rows with other ratings become `NULL`.

Sometimes that is intentional. Often it is not.

> **Analytical habit:** If unmatched rows should receive a meaningful category, write an explicit `ELSE`.


## Simple `CASE`: one column, several exact values

When every condition compares the same expression to fixed values, the simple form can be clearer:

```sql
SELECT title,
       rating,
       CASE rating
           WHEN 'G'     THEN 'General audiences'
           WHEN 'PG'    THEN 'Parental guidance'
           WHEN 'PG-13' THEN 'Parents strongly cautioned'
           WHEN 'R'     THEN 'Restricted'
           ELSE 'Other'
       END AS rating_label
FROM film;
```

Use searched `CASE` for ranges or more complex conditions. Use simple `CASE` for direct value mapping.


## `CASE` can combine several columns

A searched `CASE` is not limited to one column.

```sql
SELECT title,
       rental_rate,
       replacement_cost,
       CASE
           WHEN rental_rate >= 4 AND replacement_cost >= 25
               THEN 'High-value premium'
           WHEN rental_rate >= 4
               THEN 'Premium rental'
           ELSE 'Standard portfolio'
       END AS portfolio_group
FROM film;
```

The new category depends on the relationship among multiple values in the same row.


## Real-world example: service priority

Imagine a support table with `severity` and `hours_open`.

```sql
CASE
    WHEN severity = 'Critical' AND hours_open >= 2 THEN 'Escalate now'
    WHEN severity = 'Critical'                     THEN 'Monitor closely'
    WHEN hours_open >= 24                           THEN 'Overdue'
    ELSE 'Standard queue'
END
```

The same pattern appears in:

- customer segmentation,
- credit-risk bands,
- clinical triage,
- SLA monitoring,
- inventory status, and
- fraud-screening rules.

The SQL is simple. Defining the rule correctly is the harder analytical task.


## `CASE` in `ORDER BY`: business order instead of alphabetic order

Suppose you want ratings displayed from least to more restrictive:

```sql
SELECT title, rating
FROM film
ORDER BY
    CASE rating
        WHEN 'G'     THEN 1
        WHEN 'PG'    THEN 2
        WHEN 'PG-13' THEN 3
        WHEN 'R'     THEN 4
        WHEN 'NC-17' THEN 5
        ELSE 6
    END,
    title;
```

The numeric values are temporary sort keys. They do not have to appear in the final output.


## Preview: conditional aggregation

You will study aggregation in Module 7, but `CASE` frequently appears inside aggregate functions.

```sql
SELECT
    COUNT(*) AS total_films,
    SUM(CASE WHEN rental_rate >= 4 THEN 1 ELSE 0 END) AS premium_films
FROM film;
```

Each row contributes either `1` or `0`, and `SUM` adds those contributions.

Alternative pattern:

```sql
COUNT(CASE WHEN rental_rate >= 4 THEN 1 END)
```

Because `COUNT(expression)` ignores `NULL`, rows that do not meet the condition are skipped.


## Quick check 2

**Think before opening the answer.**

Why should a more specific `CASE` condition usually appear before a broader condition that also includes it?

<details>
<summary>Answer</summary>

Because `CASE` stops at the first true condition. If the broad condition is first, the more specific condition may never be reached.

</details>


## Try it: classify films by replacement cost

Write a query that returns `title`, `replacement_cost`, and a new `replacement_band`:

- below 15 → `Low`
- 15 up to but not including 25 → `Moderate`
- 25 or above → `High`

<details>
<summary>One solution</summary>

```sql
SELECT title,
       replacement_cost,
       CASE
           WHEN replacement_cost < 15 THEN 'Low'
           WHEN replacement_cost < 25 THEN 'Moderate'
           ELSE 'High'
       END AS replacement_band
FROM film
ORDER BY replacement_cost, title;
```

</details>


# Part 3: NULL Is a State, Not a Value


## What `NULL` means

`NULL` means the value is unknown or absent. It is not the same as:

- zero,
- an empty string `''`,
- the text `'NULL'`, or
- `FALSE`.

That distinction matters because functions and comparisons treat `NULL` differently from ordinary values.


## A useful mental model

Think of `NULL` as:

> “There is no known value available for this expression.”

That is why this is not a valid test:

```sql
WHERE email = NULL
```

Use:

```sql
WHERE email IS NULL
```

The same idea carries into transformation functions: decide whether missingness should remain missing or be replaced with an explicit fallback.


## `COALESCE`: first non-NULL value

Shan et al. describe `COALESCE` as scanning its arguments from left to right and returning the first one that is not `NULL`.

```sql
COALESCE(value_1, value_2, value_3, fallback)
```

Example:

```sql
SELECT customer_id,
       first_name,
       last_name,
       COALESCE(email, 'no email on file') AS email_display
FROM customer;
```

If `email` is present, the original email is returned. Otherwise, the fallback text is returned.


## A fallback chain

`COALESCE` can contain more than two arguments.

Conceptual example:

```sql
COALESCE(work_phone, mobile_phone, home_phone, 'no phone')
```

This expresses a business preference:

```text
work phone
   ↓ if missing
mobile phone
   ↓ if missing
home phone
   ↓ if missing
'no phone'
```

The argument order is therefore meaningful.


## Do not replace `NULL` automatically

Replacing every missing value is not always good cleaning.

Suppose a missing satisfaction score is replaced with `0`. A later average would treat “not answered” as “extremely dissatisfied.”

Those are different states.

Use a fallback only when it accurately represents the intended meaning of missingness.


## `NULLIF`: turn a specific value into NULL

`NULLIF(a, b)` returns:

- `NULL` if `a = b`,
- otherwise `a`.

```sql
SELECT customer_id,
       NULLIF(email, '') AS email_clean
FROM customer;
```

If an upstream system uses an empty string as “missing,” this standardizes that placeholder to actual SQL `NULL`.


## `NULLIF` as a divide-by-zero guard

Suppose a denominator might be zero:

```sql
replacement_cost / rental_rate
```

A zero rental rate would cause a division error. Convert zero to `NULL` first:

```sql
SELECT film_id,
       title,
       replacement_cost / NULLIF(rental_rate, 0) AS cost_rate_ratio
FROM film;
```

If the denominator is zero, the division result becomes `NULL` instead of raising a divide-by-zero error.


## `COALESCE` and `NULLIF` often work together

A common pattern is:

1. normalize placeholder values to `NULL`,
2. then choose how the output should display missingness.

Conceptual example:

```sql
COALESCE(NULLIF(TRIM(phone), ''), 'no phone')
```

Read inside out:

```text
TRIM(phone)
    ↓
empty string becomes NULL
    ↓
NULL becomes 'no phone'
```

Function composition is easier to reason about when you read the innermost transformation first.


## Quick check 3

**Think before opening the answer.**

What does `NULLIF(0, 0)` return? What does `NULLIF(5, 0)` return?

<details>
<summary>Answer</summary>

`NULLIF(0, 0)` returns `NULL` because the arguments are equal. `NULLIF(5, 0)` returns `5` because they are different.

</details>


# Part 4: Comparing Values Across a Row with GREATEST and LEAST


## Row-wise comparison is different from aggregation

`GREATEST` and `LEAST` compare several expressions **within the same row**.

```sql
SELECT GREATEST(3, 9, 4) AS largest,
       LEAST(3, 9, 4) AS smallest;
```

returns conceptually:

```text
largest | smallest
--------+---------
9       | 3
```

Do not confuse them with `MAX()` and `MIN()`, which usually summarize values **down a column across rows**.


## Example: apply a floor or cap

Suppose a derived score should never fall below zero:

```sql
GREATEST(calculated_score, 0)
```

Suppose a discount percentage should never exceed 100:

```sql
LEAST(discount_percent, 100)
```

Together, you can bound a value:

```sql
LEAST(GREATEST(raw_percent, 0), 100)
```

This constrains the result to the range 0–100.


## PostgreSQL behavior with NULL

In PostgreSQL, `GREATEST` and `LEAST` ignore `NULL` arguments when at least one non-NULL value is present.

```sql
SELECT GREATEST(NULL, 5, 10);  -- 10
SELECT LEAST(NULL, 5, 10);     -- 5
```

If all arguments are `NULL`, the result is `NULL`.

This behavior is useful, but it should still be an explicit analytical choice: comparing two known values is different from comparing one known value when another is missing.


## Real-world examples

Row-wise bounds appear in many systems:

- cap reimbursable expenses at a policy maximum,
- prevent a computed utilization rate from going below zero,
- select the latest available timestamp from several workflow stages,
- choose the lower of requested quantity and available inventory,
- clamp a model score to an allowed reporting range.

These functions are compact ways to encode numeric guardrails.


# Part 5: Standardizing Text


## Why text needs preparation

Two strings that look equivalent to a person may not be equivalent to the database.

```text
'Mary'
'mary'
' Mary '
'MARY'
```

If those values are grouped or joined without standardization, they may behave as different categories.

Text cleaning therefore focuses on **representation consistency**.


## Core string functions

| Function | Typical use |
|---|---|
| `TRIM(text)` | remove leading/trailing spaces |
| `LTRIM`, `RTRIM` | trim one side only |
| `UPPER(text)` | standardize comparisons to uppercase |
| `LOWER(text)` | standardize comparisons to lowercase |
| `INITCAP(text)` | presentation-style capitalization |
| `CONCAT(...)` | combine values; handles NULLs gracefully |
| `a || b` | PostgreSQL string concatenation operator |
| `LEFT(text, n)` | first `n` characters |
| `RIGHT(text, n)` | last `n` characters |
| `SUBSTRING(...)` | extract part of a string |
| `REPLACE(...)` | substitute one substring for another |
| `LENGTH(text)` | inspect string length |


## `TRIM`: remove invisible noise

```sql
SELECT customer_id,
       TRIM(first_name) AS first_name_clean,
       TRIM(last_name) AS last_name_clean
FROM customer;
```

Whitespace is easy to miss in a report but can break equality comparisons or create duplicate-looking categories.

A good rule is to standardize before comparing or grouping text from messy sources.


## `UPPER` and `LOWER`: make comparisons consistent

Conceptual example:

```sql
WHERE LOWER(status) = 'completed'
```

This can match values such as:

```text
Completed
COMPLETED
completed
```

Case normalization is especially useful when the source system does not enforce a controlled list of categories.


## `INITCAP`: useful for display, not identity

```sql
SELECT INITCAP(TRIM(first_name)) AS first_name_display
FROM customer;
```

`INITCAP` is useful for presentation, but do not assume it produces a culturally or linguistically correct personal name in every case.

Cleaning for **display** and cleaning for **matching** are different goals.


## Concatenation: `CONCAT` versus `||`

Shan et al. introduce both forms.

```sql
SELECT CONCAT(first_name, ' ', last_name) AS full_name
FROM customer;
```

```sql
SELECT first_name || ' ' || last_name AS full_name
FROM customer;
```

In PostgreSQL, `CONCAT` treats `NULL` arguments as empty strings, while `||` propagates `NULL`.

That means the choice can affect rows with missing name components.


## Extracting pieces of text

```sql
SELECT customer_id,
       LEFT(last_name, 1) AS last_initial,
       RIGHT(email, 3) AS email_suffix
FROM customer;
```

For more flexible positions:

```sql
SELECT SUBSTRING(title FROM 1 FOR 10) AS title_prefix
FROM film;
```

Use extraction when the position has stable meaning. Avoid parsing complex identifiers by position unless their format is actually guaranteed.


## `REPLACE` and `LENGTH` as quality tools

```sql
SELECT title,
       REPLACE(title, ' ', '_') AS title_slug,
       LENGTH(title) AS title_length
FROM film;
```

`LENGTH` is especially useful for detecting malformed values:

```sql
SELECT value
FROM some_table
WHERE LENGTH(TRIM(value)) <> expected_length;
```

A transformation function can be used not only to fix data but also to **diagnose** it.


## Compose text functions deliberately

```sql
SELECT customer_id,
       INITCAP(TRIM(first_name)) AS first_name_clean,
       UPPER(LEFT(TRIM(last_name), 1)) AS last_initial
FROM customer;
```

Read the second expression from the inside out:

```text
last_name
   ↓ TRIM
remove edge spaces
   ↓ LEFT(..., 1)
keep first character
   ↓ UPPER
standardize case
```


## Quick check 4

**Think before opening the answer.**

Why might `CONCAT(first_name, ' ', last_name)` be safer than `first_name || ' ' || last_name` when one component can be NULL?

<details>
<summary>Answer</summary>

In PostgreSQL, `CONCAT` ignores NULL arguments, whereas the `||` operator returns NULL if one concatenated expression is NULL.

</details>


# Part 6: Type Casting — Making the Type Match the Meaning


## Data types determine legal operations

A value can contain the right information but still have the wrong type.

Examples:

```text
'4.99'       -- text that represents a number
'2026-09-05' -- text that represents a date
42           -- integer that may need text presentation
```

Before comparing, calculating, or applying a type-specific function, make sure the value has the correct type.


## Two PostgreSQL casting forms

SQL-standard form:

```sql
CAST(expression AS data_type)
```

PostgreSQL shorthand:

```sql
expression::data_type
```

Examples:

```sql
SELECT CAST('4.99' AS numeric) AS rate_number,
       '2026-09-05'::date AS report_date;
```

Both forms are valid in PostgreSQL.


## Example from the textbook pattern: number to text

A numeric year cannot be concatenated as text until it is converted.

```sql
SELECT 'Year of ' || CAST(2026 AS text) AS label;
```

or:

```sql
SELECT 'Year of ' || 2026::text AS label;
```

Casting changes the expression's type for the query result; it does not permanently change the source column.


## Cast before arithmetic

If a numeric value arrives as text, convert it before calculation:

```sql
SELECT CAST('4.99' AS numeric) * 2 AS doubled_rate;
```

Do not rely on the database to guess every conversion. Implicit casts vary by operation and database system, and some ambiguous operations correctly fail.


## Invalid casts fail

This conversion is meaningful:

```sql
SELECT '2026-09-05'::date;
```

This one is not:

```sql
SELECT 'not-a-date'::date;
```

PostgreSQL raises an error because the text cannot be interpreted as a date.

> **Cleaning lesson:** Casting does not repair malformed values. Validate or standardize the value first.


## Precision matters

This query deliberately removes the decimal portion:

```sql
SELECT amount,
       amount::integer AS whole_amount
FROM payment;
```

A cast can therefore lose information.

Before converting a type, ask:

- Will precision be lost?
- Will formatting change?
- Can every source value be converted?
- Is the target type appropriate for future calculations?


## `CAST()` versus `::`

Use `CAST()` when portability and explicitness matter:

```sql
CAST(payment_date AS date)
```

Use `::` when writing concise PostgreSQL-specific SQL:

```sql
payment_date::date
```

The important skill is not choosing one syntax. It is recognizing when the current type does not support the operation you need.


## Quick check 5

**Think before opening the answer.**

Does `payment_date::date` permanently change the `payment_date` column in the database?

<details>
<summary>Answer</summary>

No. It creates a date-typed expression in the query result. The stored column remains unchanged.

</details>


# Part 7: Working with Dates and Timestamps


## Dates are analytical dimensions

Almost every business process has a time component:

- purchase date,
- rental date,
- payment timestamp,
- hire date,
- claim date,
- delivery date,
- service start and end time.

Raw timestamps are often too detailed for reporting. Date functions let you reshape them into useful periods and components.


## Current date and time

PostgreSQL provides values such as:

```sql
SELECT CURRENT_DATE,
       CURRENT_TIME,
       CURRENT_TIMESTAMP;
```

These are useful for relative calculations such as age, tenure, lateness, and “days since” measures.

Remember that results based on the current time change whenever the query is run.


## Extract a date component

```sql
SELECT payment_id,
       payment_date,
       EXTRACT(YEAR FROM payment_date) AS payment_year,
       EXTRACT(MONTH FROM payment_date) AS payment_month
FROM payment;
```

Equivalent function-style form:

```sql
DATE_PART('month', payment_date)
```

Use extraction when the individual component itself is analytically meaningful.


## `DATE_TRUNC`: move a timestamp to a reporting period

```sql
SELECT payment_id,
       payment_date,
       DATE_TRUNC('month', payment_date) AS payment_month
FROM payment;
```

Every timestamp in the same month maps to the same month-start timestamp.

Conceptually:

```text
2022-03-04 10:12  ┐
2022-03-18 17:45  ├─> 2022-03-01 00:00
2022-03-31 08:01  ┘
```

This is especially useful before `GROUP BY` in Module 7.


## `EXTRACT` and `DATE_TRUNC` answer different questions

`EXTRACT(MONTH FROM payment_date)` returns a number such as `3`.

`DATE_TRUNC('month', payment_date)` returns a timestamp representing the month, such as `2022-03-01 00:00:00`.

If your data spans several years, grouping only by extracted month number would mix March 2022 with March 2023. `DATE_TRUNC('month', ...)` preserves the year as part of the period.


## Duration with `AGE`

```sql
SELECT rental_id,
       rental_date,
       return_date,
       AGE(return_date, rental_date) AS rental_duration
FROM rental
WHERE return_date IS NOT NULL;
```

`AGE` returns an interval describing the elapsed time.

For duration analysis, always consider incomplete events. A rental with no `return_date` is not a zero-day rental; it is an open or missing-end-date case.


## Real-world example: service elapsed time

Conceptual query:

```sql
SELECT ticket_id,
       opened_at,
       closed_at,
       AGE(COALESCE(closed_at, CURRENT_TIMESTAMP), opened_at) AS elapsed
FROM support_ticket;
```

Here the business rule is explicit:

- closed ticket → elapsed until closure,
- open ticket → elapsed until now.

`COALESCE` and a date function work together to define the metric.


## Day-of-week analysis

```sql
SELECT payment_id,
       payment_date,
       EXTRACT(DOW FROM payment_date) AS day_of_week
FROM payment;
```

In PostgreSQL, `DOW` uses numeric day values. For a report, you may translate those numbers with `CASE` or a formatting function later.

This illustrates a common workflow:

```text
raw timestamp
   ↓
extract component
   ↓
map component to business label
```


## Quick check 6

**Think before opening the answer.**

Why is `DATE_TRUNC('month', payment_date)` usually safer than `EXTRACT(MONTH FROM payment_date)` when grouping transactions across multiple years?

<details>
<summary>Answer</summary>

Because `DATE_TRUNC` retains the year in the month period. `EXTRACT(MONTH ...)` returns only 1–12, so the same month number from different years can be combined unintentionally.

</details>


# Part 8: DISTINCT and the Difference Between Cleaning and Hiding Problems


## `DISTINCT` returns unique result rows

```sql
SELECT DISTINCT rating
FROM film
ORDER BY rating;
```

With several columns:

```sql
SELECT DISTINCT rating, rental_rate
FROM film
ORDER BY rating, rental_rate;
```

The second query returns unique **combinations** of rating and rental rate.


## `DISTINCT` changes the result grain

Before:

```text
one row per film
```

After:

```sql
SELECT DISTINCT rating
FROM film;
```

The grain becomes:

```text
one row per distinct rating value
```

That is a meaningful analytical transformation, not merely a formatting option.


## Do not use `DISTINCT` as a mystery-duplicate eraser

Suppose a join unexpectedly returns each customer many times.

This may hide the symptom:

```sql
SELECT DISTINCT c.customer_id, c.first_name, c.last_name
FROM customer c
JOIN rental r ON c.customer_id = r.customer_id;
```

But the repeated rows were not necessarily an error. One customer can have many rentals.

Before adding `DISTINCT`, ask:

> What is the grain after the join, and why are there multiple rows?


## Diagnostic sequence for duplicates

When rows repeat unexpectedly:

1. identify the grain of every source table,
2. identify the join cardinality,
3. verify the `ON` condition,
4. inspect a single known ID,
5. decide whether duplicates represent real one-to-many relationships,
6. only then decide whether deduplication is appropriate.

`DISTINCT` should express an intentional request for unique rows, not conceal a data-model misunderstanding.


# Part 9: Building a Cleaning Pipeline


## One expression can become hard to audit

You can nest many functions in one line:

```sql
COALESCE(INITCAP(NULLIF(TRIM(raw_name), '')), 'Unknown')
```

This may be correct, but a long query with many such expressions becomes difficult to debug.

Module 5 introduced CTEs. They are an excellent structure for cleaning because each named step can do one job.


## A four-step pipeline pattern

Imagine a staging table with messy text fields.

```text
raw rows
  ↓
trim whitespace
  ↓
standardize missing values
  ↓
cast types
  ↓
apply business categories
  ↓
analysis-ready rows
```

Each arrow can become a named CTE.


## CTE cleaning example — Step 1

```sql
WITH trimmed AS (
    SELECT customer_id,
           TRIM(first_name) AS first_name,
           TRIM(last_name)  AS last_name,
           NULLIF(TRIM(email), '') AS email
    FROM customer
)
SELECT *
FROM trimmed;
```

Test the first step before adding the next one.


## CTE cleaning example — Step 2

```sql
WITH trimmed AS (
    SELECT customer_id,
           TRIM(first_name) AS first_name,
           TRIM(last_name)  AS last_name,
           NULLIF(TRIM(email), '') AS email
    FROM customer
),
standardized AS (
    SELECT customer_id,
           INITCAP(first_name) AS first_name,
           INITCAP(last_name)  AS last_name,
           COALESCE(email, 'no email on file') AS email
    FROM trimmed
)
SELECT *
FROM standardized;
```

The second step does not have to repeat the trimming logic. It consumes the already-cleaned result.


## Pipeline principle: one transformation family per step

A useful naming pattern is:

```text
raw_source
trimmed_text
normalized_missing
typed_values
classified_rows
final_output
```

Names should describe **what the step produces**, not merely `cte1`, `cte2`, and `temp`.

This makes the query readable even before someone examines each expression.


## Validate every stage

A cleaning pipeline should include checks.

Examples:

```sql
-- How many rows survived?
SELECT COUNT(*) FROM typed_values;

-- Did any missing values remain?
SELECT COUNT(*)
FROM normalized_missing
WHERE email IS NULL;

-- What categories were produced?
SELECT DISTINCT price_tier
FROM classified_rows;
```

A pipeline is trustworthy when each intermediate result can be explained and tested.


## Real-world pipeline: claims data

A claims analyst might receive:

```text
claim_id | service_date_text | billed_amount_text | status_raw
```

A defensible pipeline could:

1. trim and standardize `status_raw`,
2. convert blank fields to `NULL`,
3. cast valid date and amount fields,
4. create business categories with `CASE`,
5. flag impossible or incomplete records,
6. aggregate only after the grain and types are confirmed.

SQL cleaning is not a collection of isolated functions. It is a sequence of decisions about meaning.


# Part 10: Common Errors and How to Diagnose Them


## Error 1: overlapping `CASE` conditions in the wrong order

**Symptom:** a specific category never appears.

**Diagnosis:** test a value that should fall into that category and evaluate each `WHEN` from top to bottom.

**Fix:** place narrower conditions before broader conditions.


## Error 2: forgetting `ELSE`

**Symptom:** unexpected `NULL` values in a derived category.

**Diagnosis:** count unmatched rows or inspect the distinct derived values.

```sql
SELECT DISTINCT
       CASE ... END AS category
FROM ...;
```

**Fix:** add an explicit `ELSE` if unmatched rows should receive a label.


## Error 3: replacing missing values with a misleading default

**Symptom:** summaries shift unexpectedly after cleaning.

Example: replacing missing numeric values with `0` changes averages.

**Fix:** decide whether missing means zero, unknown, not applicable, or something else before applying `COALESCE`.


## Error 4: casting malformed text

**Symptom:** PostgreSQL reports invalid input syntax for the target type.

**Diagnosis:** locate nonconforming values before casting.

Conceptual pattern:

```sql
SELECT raw_value
FROM staging
WHERE raw_value does_not_match_expected_format;
```

**Fix:** standardize, validate, or quarantine malformed values before type conversion.


## Error 5: using text comparison for dates or numbers

Text comparison is lexical, not numeric or chronological.

For example, as text:

```text
'100' < '20'
```

can be true because comparison proceeds character by character.

Use the type that matches the meaning before comparing.


## Error 6: treating an open event as a zero-duration event

A missing `return_date` or `closed_at` usually means the event is incomplete or still open.

It does **not** automatically mean duration = 0.

Choose the rule explicitly:

- exclude open events,
- calculate duration through the current time,
- report them in a separate category.


## Error 7: adding `DISTINCT` before understanding duplicates

**Symptom:** the query “works” after `DISTINCT`, but nobody can explain why duplicates existed.

**Fix:** trace grain and cardinality first. Deduplicate only when uniqueness is actually part of the requested result.


# Part 11: Guided Practice


## Practice 1: length categories

Return `title`, `length`, and `length_band` with:

- `< 90` → Short
- `< 120` → Medium
- otherwise → Long

<details>
<summary>Solution</summary>

```sql
SELECT title,
       length,
       CASE
           WHEN length < 90 THEN 'Short'
           WHEN length < 120 THEN 'Medium'
           ELSE 'Long'
       END AS length_band
FROM film
ORDER BY length, title;
```

</details>


## Practice 2: missing email display

Return each customer's ID and email, displaying `no email on file` when the value is missing.

<details>
<summary>Solution</summary>

```sql
SELECT customer_id,
       COALESCE(email, 'no email on file') AS email_display
FROM customer;
```

</details>


## Practice 3: safe ratio

Return film title and the ratio of replacement cost to rental rate. Protect against a zero denominator.

<details>
<summary>Solution</summary>

```sql
SELECT title,
       replacement_cost / NULLIF(rental_rate, 0) AS cost_rate_ratio
FROM film;
```

</details>


## Practice 4: standardized names

Return customer ID, a cleaned first name using `TRIM` and `INITCAP`, and the first letter of the last name.

<details>
<summary>Solution</summary>

```sql
SELECT customer_id,
       INITCAP(TRIM(first_name)) AS first_name_clean,
       LEFT(TRIM(last_name), 1) AS last_initial
FROM customer;
```

</details>


## Practice 5: payment period fields

Return payment ID, payment timestamp, reporting month, and day-of-week number.

<details>
<summary>Solution</summary>

```sql
SELECT payment_id,
       payment_date,
       DATE_TRUNC('month', payment_date) AS payment_month,
       EXTRACT(DOW FROM payment_date) AS day_of_week
FROM payment;
```

</details>


## Practice 6: distinct rating/rate combinations

Return every unique combination of film rating and rental rate.

<details>
<summary>Solution</summary>

```sql
SELECT DISTINCT rating, rental_rate
FROM film
ORDER BY rating, rental_rate;
```

</details>


## Practice 7: row-wise bounds

Conceptually, a calculated percentage called `raw_percent` must be kept between 0 and 100.

Write the expression.

<details>
<summary>Solution</summary>

```sql
LEAST(GREATEST(raw_percent, 0), 100)
```

The inner `GREATEST` applies the floor of 0; the outer `LEAST` applies the cap of 100.

</details>


# Part 12: AI Critique Practice


## Critique 1: Does this classification work?

Prompt:

> Label films under 90 minutes Short, films from 90 through 119 Medium, and all others Long.

AI-generated SQL:

```sql
SELECT title,
       CASE
           WHEN length < 120 THEN 'Medium'
           WHEN length < 90 THEN 'Short'
           ELSE 'Long'
       END AS length_band
FROM film;
```

What is wrong?

<details>
<summary>Critique</summary>

The conditions overlap and are ordered incorrectly. Every length below 90 also satisfies `< 120`, so those rows are labeled Medium before the Short condition is reached.

Correct order:

```sql
CASE
    WHEN length < 90 THEN 'Short'
    WHEN length < 120 THEN 'Medium'
    ELSE 'Long'
END
```

</details>


## Critique 2: Is this missing-value fix defensible?

AI-generated SQL:

```sql
SELECT AVG(COALESCE(score, 0))
FROM survey_response;
```

The prompt said: “Calculate the average score; some respondents did not answer.”

<details>
<summary>Critique</summary>

Probably not. Replacing an unanswered score with zero changes the meaning of the average by treating missing responses as actual zero scores. If missing means “no response,” `AVG(score)` already ignores NULL values and is usually the more appropriate calculation. The business definition must be confirmed before substitution.

</details>


## Critique 3: Why is this deduplication suspicious?

AI-generated SQL:

```sql
SELECT DISTINCT c.customer_id, c.first_name, c.last_name
FROM customer c
JOIN rental r ON c.customer_id = r.customer_id;
```

The prompt said: “List every rental with the customer name.”

<details>
<summary>Critique</summary>

The requested grain is one row per rental, but the `SELECT` list contains only customer columns and `DISTINCT` collapses repeated customers. The query therefore loses rental-level information. It should include rental identifiers/details and should not deduplicate valid one-to-many rows.

</details>


## A reliable AI-review checklist for transformations

When reviewing generated SQL, ask:

1. Is the business rule translated correctly?
2. Are `CASE` conditions mutually exclusive or intentionally ordered?
3. What happens to `NULL`?
4. Could a cast fail or lose precision?
5. Does text standardization alter meaning?
6. Do date functions preserve the intended period?
7. Did `DISTINCT` hide legitimate repeated rows?
8. Is the grain unchanged unless the question requires it to change?
9. Can each transformation step be tested independently?


# Part 13: Knowledge Check


## Knowledge check 1

**Think before opening the answer.**

What does a `CASE` expression return when no `WHEN` is true and there is no `ELSE`?

<details>
<summary>Answer</summary>

`NULL`.

</details>


## Knowledge check 2

**Think before opening the answer.**

Which function returns the first non-NULL argument from left to right?

<details>
<summary>Answer</summary>

`COALESCE`.

</details>


## Knowledge check 3

**Think before opening the answer.**

Which function is commonly used to turn a zero denominator into NULL before division?

<details>
<summary>Answer</summary>

`NULLIF(denominator, 0)`.

</details>


## Knowledge check 4

**Think before opening the answer.**

Which function would you use to cap a value at 100?

<details>
<summary>Answer</summary>

`LEAST(value, 100)`.

</details>


## Knowledge check 5

**Think before opening the answer.**

What is the main difference between `MAX()` and `GREATEST()`?

<details>
<summary>Answer</summary>

`MAX()` is typically an aggregate across rows in a column or group; `GREATEST()` compares multiple expressions within one row.

</details>


## Knowledge check 6

**Think before opening the answer.**

What is the difference between `CAST(x AS date)` and `x::date` in PostgreSQL?

<details>
<summary>Answer</summary>

They are equivalent type casts in PostgreSQL. `CAST()` is SQL-standard syntax; `::` is PostgreSQL shorthand.

</details>


## Knowledge check 7

**Think before opening the answer.**

Why can grouping by `EXTRACT(MONTH FROM ts)` be dangerous across several years?

<details>
<summary>Answer</summary>

It returns only the month number 1–12, so the same month from different years can be combined.

</details>


## Knowledge check 8

**Think before opening the answer.**

What does `DISTINCT a, b` make unique?

<details>
<summary>Answer</summary>

The combination of columns `a` and `b`, not each column independently.

</details>


# Part 14: Concept Map


## Module 6 concept map

```text
RAW / INCONVENIENT VALUES
        |
        +--> conditional meaning
        |       `CASE`
        |
        +--> missing / exceptional values
        |       `COALESCE`
        |       `NULLIF`
        |
        +--> row-wise bounds
        |       `GREATEST`
        |       `LEAST`
        |
        +--> inconsistent text
        |       `TRIM`, `UPPER`, `LOWER`, `INITCAP`
        |       `CONCAT`, `LEFT`, `RIGHT`, `SUBSTRING`
        |       `REPLACE`, `LENGTH`
        |
        +--> wrong data type
        |       `CAST(... AS type)`
        |       `::type`
        |
        +--> timestamp detail
        |       `DATE_TRUNC`
        |       `EXTRACT` / `DATE_PART`
        |       `AGE`
        |
        +--> duplicate result rows
                `DISTINCT` -- only when uniqueness is intended

        ↓
CLEANER, TYPED, EXPLAINABLE DATA
        ↓
GROUP BY / AGGREGATION / REPORTING
```


# Part 15: Lab 5 Readiness


## Before starting Lab 5, make sure you can do these without copying

- write a searched `CASE` from a plain-language rule
- explain what happens at every boundary value
- predict the output when no `CASE` condition matches
- write a `COALESCE` fallback chain
- use `NULLIF` to normalize a placeholder or prevent divide-by-zero
- distinguish `GREATEST`/`LEAST` from `MAX`/`MIN`
- clean a text field using more than one string function
- write both forms of a type cast
- explain why an invalid cast fails
- derive month and date parts from a timestamp
- use `DISTINCT` and state exactly what becomes unique
- combine transformations into readable steps and test intermediate results


## Lab habit: validate the edge cases first

When your query appears to work, deliberately inspect:

- the shortest and longest values,
- exact band boundaries,
- `NULL` values,
- zeros before division,
- empty strings,
- mixed-case strings,
- dates near month/year boundaries,
- and repeated rows.

A transformation is not validated by the typical row. It is validated by the rows most likely to expose a hidden assumption.


# Part 16: Quiz 1 Connection


## Module 6 also closes the first foundation sequence

Quiz 1 covers the earlier foundations through Module 5. While Module 6 introduces a new transformation toolkit, many of its habits reinforce the same reasoning skills:

```text
Know the grain.
Know the data type.
Know how NULL behaves.
Know which rows survive each clause.
Know whether repeated rows are expected.
Translate the business question before writing syntax.
```

These habits matter more than memorizing isolated functions.


# Part 17: Key Takeaways


## Key takeaways

1. **`CASE` encodes conditional business logic.** The first true condition wins, so ordering and boundary definitions matter.
2. **Missingness has meaning.** Use `COALESCE` and `NULLIF` deliberately rather than replacing values automatically.
3. **`GREATEST` and `LEAST` compare expressions within a row.** They are useful for floors, caps, and guardrails.
4. **Text standardization improves comparability.** Trim whitespace, normalize case, and use extraction/replacement functions only when the source format supports those rules.
5. **Types control operations.** Cast values before calculations or comparisons when the stored type does not match the analytical meaning.
6. **Dates become more useful when reshaped into periods and components.** `DATE_TRUNC`, `EXTRACT`, `DATE_PART`, and `AGE` support common reporting questions.
7. **`DISTINCT` should express intentional uniqueness.** It is not a substitute for diagnosing unexpected duplicates after a join.
8. **Cleaning is a pipeline.** Named CTE steps make multi-stage transformations easier to test, review, and trust.


## Looking ahead

Module 7 moves from preparation to **aggregation**.

The transformations from this module become inputs to:

- `COUNT`, `SUM`, `AVG`, `MIN`, and `MAX`,
- `GROUP BY`,
- `HAVING`,
- grouping sets, and
- conditional aggregation.

Examples:

```sql
DATE_TRUNC('month', payment_date)
```

becomes a grouping key, while:

```sql
CASE WHEN ... THEN ... END
```

becomes a conditional measure or category.

Clean values first; summarize second.


# References


## Principal source

Shan, J., Li, H., Goldwasser, M., Malik, U., & Johnston, B. (2025). *SQL for Data Analytics: Analyze Data Effectively, Uncover Insights and Master Advanced SQL for Real-World Applications* (4th ed.). Packt Publishing. **Chapter 6: Transforming and Updating Data.**

## Supporting references

PostgreSQL Global Development Group. (n.d.). *PostgreSQL 16 Documentation: Conditional expressions.*

PostgreSQL Global Development Group. (n.d.). *PostgreSQL 16 Documentation: String functions and operators.*

PostgreSQL Global Development Group. (n.d.). *PostgreSQL 16 Documentation: Date/time functions and operators.*

PostgreSQL Global Development Group. (n.d.). *PostgreSQL 16 Documentation: Value expressions and type casts.*

## Course practice database

Pagila PostgreSQL sample database: `film`, `customer`, `payment`, and `rental` tables are used throughout the examples.
